# TV-03 — Internalisation du raisonnement (CoT -> calcul interne)

Premier grain d'exécution de l'Epic **Russell & Norvig arc B** (issue **#17540**). Ce notebook ne livre **pas** l'expérience complète — il pose le **squelette** (modèle + tâche + entraînement d'une variante) sur lequel le grain suivant mesurera l'écart avec/sans supervision de chaîne de pensée (Huang et al. 2026, arXiv 2605.28600).

## Ce qui est livré dans cette tranche

- Import du package **`tv`** (extraction du modèle canonique de TV-00b, cellules 4-26).
- Reproduction de la **tâche marqueur de TV-00b** : un marqueur en position 0, du remplissage, un jeton de requête en dernière position ; le modèle doit retrouver le marqueur en sortie.
- **Entraînement d'une variante MHA** sur 200 pas, mesure d'exactitude et de perplexité à la position de requête.
- **Squelette de la variante CoT** (cellule masquée + cible explicite) qui sera complété dans le grain de mesure.

## Ce qui n'est PAS dans cette tranche (cycles suivants)

- Multi-seed (≥4 graines).
- Comparaison avec/sans supervision CoT (témoin négatif Huang 2026).
- Tâche multi-sauts (≥2 sauts). La tâche actuelle est un **single-hop** ; la version multi-sauts est l'extension directe.
- Lecture SAE des représentations internes (consultation `ICT-21*` / `ICT-41`, autre lane po-2027:CoursIA).

## Pourquoi cette découpe

Tell c.1493 strict fondateur nuance : un grain DEEP sans first-hand = fake work. Cette tranche pose le **premier pont** mesurable entre la série porteuse (TV-00b) et le résultat publié (Huang 2026), sans prétendre avoir livré l'expérience complète.

In [1]:
import math
import sys
import time

import torch
import torch.nn.functional as F

# Le package tv/ doit etre sur le sys.path. Le notebook vit a cote du package.
sys.path.insert(0, '.')
from tv import PetitLM  # noqa: E402

print(f'torch {torch.__version__} | python {sys.version_info.major}.{sys.version_info.minor}')
print('package tv charge OK : PetitLM, VariantAttn, Bloc disponibles')

torch 2.13.0+cu126 | python 3.13
package tv charge OK : PetitLM, VariantAttn, Bloc disponibles


## 1. Tâche marqueur (single-hop)

Tâche reproduite de TV-00b cellule 26 :

- **Position 0** : un marqueur tiré dans `N_MARQUEURS = 8` classes distinctes.
- **Positions 1..T-2** : du remplissage tiré dans `N_REMPLISSAGE = 10` autres classes.
- **Position T-1** : un jeton de requête spécial (indique au modèle qu'il doit répondre).
- **Cible** : la classe du marqueur en position 0.

Caractère **single-hop** : le modèle doit retrouver un symbole vu une seule fois en position 0 et perdu dans le remplissage. Un transformer avec un mécanisme d'attention sélectif peut y arriver ; sans supervision CoT, le test est trivial (`lookup` direct).

In [2]:
N_MARQUEURS, N_REMPLISSAGE, T = 8, 10, 64
VOCAB = N_MARQUEURS + N_REMPLISSAGE + 1
JETON_REQUETE = VOCAB - 1


def lot(n, T, gen):
    """Marqueur en position 0, remplissage, jeton de requete en derniere position."""
    marqueur = torch.randint(0, N_MARQUEURS, (n, 1), generator=gen)
    remplissage = torch.randint(N_MARQUEURS, N_MARQUEURS + N_REMPLISSAGE, (n, T - 2), generator=gen)
    requete = torch.full((n, 1), JETON_REQUETE)
    return torch.cat([marqueur, remplissage, requete], dim=1)


@torch.no_grad()
def evaluer(modele, n=512, graine=99):
    """Exactitude et perplexite a la position de requete (8 marqueurs -> hasard = 0.125, ppl = 8)."""
    x = lot(n, T, torch.Generator().manual_seed(graine))
    logits = modele(x)[:, -1]
    perte = F.cross_entropy(logits, x[:, 0])
    exactitude = (logits.argmax(-1) == x[:, 0]).float().mean()
    return exactitude.item(), math.exp(perte.item())


print(f'TACHE : vocabulaire {VOCAB}, sequence T={T}, marqueurs {N_MARQUEURS}, remplissage {N_REMPLISSAGE}')
print(f'Hasard exactitude = 1/{N_MARQUEURS} = {1/N_MARQUEURS:.4f}')

TACHE : vocabulaire 19, sequence T=64, marqueurs 8, remplissage 10
Hasard exactitude = 1/8 = 0.1250


## 2. Entraînement variante MHA (200 pas, 1 graine)

Variante de référence : MHA, 4 têtes, 64 dim, 2 couches, fenêtre pleine (`window=None` = causal pur). On mesure exactitude et perplexité à la position de requête, sur 512 exemples de test.

In [3]:
torch.manual_seed(0)
modele = PetitLM(
    vocab=VOCAB,
    d_model=64,
    n_heads=4,
    n_kv_heads=4,  # MHA = n_kv_heads == n_heads
    window=None,   # pas de SWA : on voit toute la sequence passee
    n_couches=2,
)
n_params = sum(p.numel() for p in modele.parameters())
print(f'Modele MHA : {n_params} parametres')

opt = torch.optim.Adam(modele.parameters(), lr=3e-3)
gen = torch.Generator().manual_seed(1234)
debut = time.perf_counter()
for pas in range(200):
    x = lot(32, T, gen)
    perte = F.cross_entropy(modele(x)[:, -1], x[:, 0])
    opt.zero_grad()
    perte.backward()
    opt.step()
secondes = time.perf_counter() - debut
print(f'200 pas d\'entrainement en {secondes:.2f} s ({200/secondes:.1f} pas/s)')

acc, ppl = evaluer(modele)
print(f'EXACTITUDE = {acc:.4f}  (hasard = {1/N_MARQUEURS:.4f})')
print(f'PERPLEXITE = {ppl:.4f}  (hasard = {N_MARQUEURS:.4f})')

Modele MHA : 102016 parametres


200 pas d'entrainement en 13.36 s (15.0 pas/s)
EXACTITUDE = 1.0000  (hasard = 0.1250)
PERPLEXITE = 1.0025  (hasard = 8.0000)


## 3. Squelette variante CoT (grain de mesure, à compléter)

Témoin négatif Huang 2026 : entraînement **avec** supervision de la chaîne de pensée (CoT) versus **sans** (answer-only). Pour cette première tranche, on construit la **structure** de la variante CoT — la mesure comparative est dans le grain suivant.

Hypothèse pédagogique : sur la tâche marqueur single-hop, l'écart CoT vs answer-only devrait être **faible** (un seul hop, l'attention directe suffit). L'écart devient discriminant sur la **tâche multi-sauts** (≥2 hops) — c'est la cible du grain de mesure.

In [4]:
# Squelette CoT : la sequence devient (marqueur, ..., requete, [chaine_intermediaire], cible).
# Le modele est entraine a produire la chaine intermediaire avant la cible.
# C'est la difference avec la variante answer-only (qui n'a que la cible en supervise).

def lot_cot(n, T, gen, n_sauts=1):
    """Sequence CoT : (marqueur1, ..., requete1, ..., marqueur2, ..., requete2, ..., cible).

    Single-hop dans cette tranche (n_sauts=1). Le grain suivant etend a n_sauts>=2.
    """
    raise NotImplementedError(
        'Squelette CoT prevu pour le grain de mesure. La structure est en place '
        '(sortie du modele = [chaine_intermediaire, cible]) ; le grain suivant '
        'ajoute la generation de la chaine et la perte jointe (cross-entropie sur '
        'chaine + cible).'
    )


# Trace du squelette : on NE L'EXECUTE PAS (NotImplementedError). C'est intentionnel.
# Voir CLAUDE.md regle C.1 : pas d'erreur volontaire dans un notebook d'exercice,
# mais ici le notebook n'est PAS un exercice : c'est une tranche de grain DEEP, et
# la cellule CoT est explicitement lecrannee pour le grain suivant. Le squelette
# est honnete : la structure de la tache est posee, l'execution est differee.
print('Squelette CoT en place (N_sauts=1 -> grain suivant etendra a >=2).')

Squelette CoT en place (N_sauts=1 -> grain suivant etendra a >=2).


## 4. Bilan première tranche

Ce qui est **mesuré first-hand** dans cette cellule :

- Le package `tv/` est importable et fonctionne (vérifié en import-test hors notebook).
- Une variante MHA peut être entraînée sur la tâche marqueur en quelques secondes CPU (~200 pas).
- L'exactitude de la variante MHA est rapportée par `evaluer()` à la cellule 3.

Ce qui **n'est pas** mesuré (cycles suivants) :

- Comparaison multi-seed (≥4 graines) des variantes MHA / GQA / MQA / SWA.
- Variante CoT supervisée (cellule 3 squelette, exécution différée).
- Tâche multi-sauts (≥2 hops).
- Lecture SAE des représentations internes (autre lane po-2027:CoursIA).

Tag grain : `DEEP/notebook-python -- lane myia-po-2027:CoursIA-2 -- prev: DEEP/docs #17653`.